# 🏥 Tech Challenge IADT - Fase 3: Assistente Virtual Médico
Este notebook automatiza a **pipeline completa** do projeto conforme documentação oficial:

### 📋 Etapas da Pipeline
1. **Etapa 1**: Coleta e preparação de dados médicos (Scraping + Anonimização)
2. **Etapa 2**: Base de prontuários simulada (SQLite)
3. **Etapa 3**: Fine-tuning do LLM (BioMistral/TinyLlama + LoRA)
4. **Etapa 4**: Sistema RAG (Retrieval-Augmented Generation)
5. **Etapa 5**: Workflow LangGraph (orquestração de estados)
6. **Etapa 6**: Assistente médico completo (LangChain + RAG + Prontuários + Citações)

In [1]:
import os

# Detecção robusta: Só tenta montar se for REALMENTE o Google Colab oficial
IS_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/google/auth/external/interactive_login')

if IS_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("✅ Google Drive montado com sucesso.")
        base_path = "/content"
    except Exception as e:
        print(f"⚠️ Ambiente Colab, mas erro ao montar Drive: {e}")
        base_path = "/content"
else:
    # Se estiver no Kaggle ou Local, usamos o caminho relativo ou do Kaggle
    print("🏠 Ambiente: Kaggle/Local detectado. Ignorando Google Drive.")
    # No Kaggle, o diretório correto para salvar arquivos é /kaggle/working
    base_path = "/kaggle/working" if os.path.exists("/kaggle/working") else os.getcwd()

os.chdir(base_path)
print(f"📍 Diretório de trabalho definido para: {os.getcwd()}")

# Garante que as pastas do projeto existam no ambiente atual
repo_name = 'tech-challenge-fase-3'
if not os.path.exists(repo_name):
    print(f"📥 Clonando repositório em {os.getcwd()}...")
    !git clone https://github.com/vagnerbarbosa/tech-challenge-fase-3.git

# Entra na pasta do projeto se ela existir
if os.path.exists(repo_name):
    os.chdir(repo_name)
    !mkdir -p data/raw data/processed models/checkpoints logs
    print(f"📂 Pasta do projeto acessada: {os.getcwd()}")

🏠 Ambiente: Kaggle/Local detectado. Ignorando Google Drive.
📍 Diretório de trabalho definido para: /kaggle/working
📂 Pasta do projeto acessada: /kaggle/working/tech-challenge-fase-3


In [ ]:
# 1. Limpeza e Instalação Corrigida para Kaggle (Numpy 2.x compatível)
print("⏳ Iniciando configuração do ambiente no Kaggle...")
print("🔄 Removendo pacotes conflitantes...")

# Desinstalamos versões antigas que causam o 'ValueError: numpy.dtype size changed'
!pip uninstall -y numpy pandas datasets transformers trl langchain langchain-community -q

print("📥 Instalando dependências compatíveis...")
!pip install -q --upgrade pip

# Instalamos Numpy e Pandas sem fixar versão 1.x para evitar incompatibilidade binária
!pip install -q numpy pandas datasets

# Mantemos as versões específicas das bibliotecas de IA do seu projeto
!pip install -q \
    transformers==4.46.0 \
    tokenizers==0.20.0 \
    sentence-transformers==3.0.1 \
    accelerate==0.34.2 \
    peft \
    bitsandbytes \
    trl==0.14.0 \
    safetensors \
    sentencepiece

# Orquestradores (LangChain e LangGraph)
!pip install -q \
    langchain==0.2.14 \
    langchain-core==0.2.35 \
    langchain-community==0.2.12 \
    langchain-huggingface==0.0.3 \
    langchain-text-splitters==0.2.2 \
    langgraph==0.2.16

# Outras dependências
!pip install -q requests==2.32.4 pyarrow python-dotenv rich tqdm beautifulsoup4 graphviz

# --- Configuração do Repositório ---
import os

# No Kaggle, usamos obrigatoriamente /kaggle/working para escrita
base_path = "/kaggle/working"
os.chdir(base_path)

repo_name = 'tech-challenge-fase-3'
repo_url = 'https://github.com/vagnerbarbosa/tech-challenge-fase-3.git'

# 2. Clonagem inteligente (requer Internet ON nas Settings do Kaggle)
if not os.path.exists(os.path.join(base_path, repo_name)):
    print(f"📥 Clonando repositório {repo_name}...")
    !git clone {repo_url}
else:
    print(f"✅ Repositório {repo_name} já existe.")

# 3. Definição do diretório de execução e criação de pastas
project_path = os.path.join(base_path, repo_name)
if os.path.exists(project_path):
    os.chdir(project_path)
    !mkdir -p data/raw data/processed models/checkpoints logs
    print(f"📍 Diretório de trabalho: {os.getcwd()}")

print("\n✅ Instalação concluída!")
print("⚠️ O kernel será reiniciado automaticamente em 5 segundos para aplicar as mudanças.")

# 4. Reinicialização forçada do Kernel (Essencial para resolver o erro do Numpy no Kaggle)
import time
time.sleep(5)
os._exit(0)

⏳ Iniciando configuração do ambiente no Kaggle...
🔄 Removendo pacotes conflitantes...
📥 Instalando dependências compatíveis...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 3.0.1 requires transformers<5.0.0,>=4.34.0, which is not installed.
langchain-huggingface 0.0.3 requires transformers>=4.39.0, which is not installed.
kaggle-environments 1.27.0 requires transformers>=4.33.1, which is not installed.
peft 0.18.1 requires transformers, which is not installed.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.1 requires numpy<2.4,>=1.22, but you have numpy 2.4.3 which is incompatible.
ydata-profiling 4.18.1 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12

In [1]:
import os
import sys

# Define o diretório onde o repositório foi clonado no Kaggle
project_path = "/kaggle/working/tech-challenge-fase-3"

if os.path.exists(project_path):
    os.chdir(project_path)
    # Adiciona a pasta src ao path do sistema para permitir os imports
    sys.path.append(os.path.join(project_path, "src"))
    print(f"✅ Pronto! Diretório definido para: {os.getcwd()}")
else:
    print("❌ Erro: Pasta do projeto não encontrada. Certifique-se de que o clone funcionou.")

✅ Pronto! Diretório definido para: /kaggle/working/tech-challenge-fase-3


## 📂 ETAPA 1: Coleta e Preparação de Dados Médicos
**Objetivo:** Extrair, anonimizar e preparar dados médicos para fine-tuning.

- Web scraping de 3 fontes: CONITEC/MS, TelessaúdeRS, RadReport
- Pré-processamento e curadoria com anonimização (LGPD)
- Validação e limpeza dos dados
- Unificação em dataset para treinamento

In [2]:
import os
import sys
from src.fine_tuning.data_preparation import DataPreparation
from src.utils.logging_config import setup_logging, get_logger

# Configura logging
setup_logging()
logger = get_logger(__name__)

print("="*60)
print("📊 ETAPA 1: Scraping e Processamento de Dados")
print("="*60)

prep = DataPreparation(data_path="./data")
dataset = prep.prepare_dataset()

print(f"\n✅ Etapa 1 concluída!")
print(f"Total de exemplos coletados/processados: {len(dataset)}")
print("\n📄 Exemplo de dado formatado:")
print("-"*60)
print(dataset[0]['text'][:800] + "...")

2026-03-18 19:28:16 | INFO     | src.utils.logging_config | Logging configurado. Arquivo: logs/medical_assistant_20260318.log
2026-03-18 19:28:16 | INFO     | src.utils.validators | DataValidator inicializado
2026-03-18 19:28:16 | INFO     | src.fine_tuning.data_preparation | DataPreparation inicializado
2026-03-18 19:28:16 | INFO     | src.fine_tuning.data_preparation |   Dados brutos: /kaggle/working/tech-challenge-fase-3/data/raw
2026-03-18 19:28:16 | INFO     | src.fine_tuning.data_preparation |   Dados processados: /kaggle/working/tech-challenge-fase-3/data/processed
2026-03-18 19:28:16 | INFO     | src.fine_tuning.data_preparation | Iniciando preparação do dataset...
2026-03-18 19:28:16 | INFO     | src.fine_tuning.data_preparation | ============================================================
2026-03-18 19:28:16 | INFO     | src.fine_tuning.data_preparation | VALIDAÇÃO DO DIRETÓRIO DE DADOS
2026-03-18 19:28:16 | INFO     | src.fine_tuning.data_preparation | =====================

📊 ETAPA 1: Scraping e Processamento de Dados

✅ Etapa 1 concluída!
Total de exemplos coletados/processados: 77

📄 Exemplo de dado formatado:
------------------------------------------------------------
### Instrução:
Como estruturar um laudo de TC de Crânio sem Contraste?

### Contexto:
Modalidade: TC (Tomografia Computadorizada) | Indicações: AVC, TCE, cefaleia aguda, alteração do nível de consciência

### Resposta:
TÉCNICA: TC de crânio sem contraste intravenoso.

ACHADOS:
Parênquima cerebral: [descrever atenuação, lesões, sinais de edema]
Sistema ventricular: [descrever tamanho e simetria]
Cisternas: [descrever perviedade]
Linha média: [descrever se centrada ou desviada]
Estruturas ósseas: [descrever integridade]
Seios paranasais e mastoides: [descrever aeração]

IMPRESSÃO:
[Resumo dos achados principais e correlação clínica]...


## 🗄️ ETAPA 2: Base de Dados Simulada de Prontuários
**Objetivo:** Criar e consultar a base SQLite com prontuários médicos fictícios.

- 5 pacientes fictícios com dados completos
- Histórico médico, medicações, alergias, exames
- Integração com o assistente para contexto personalizado

In [3]:
from src.database.patient_records import PatientDatabase

print("="*60)
print("🗄️ ETAPA 2: Base de Prontuários Simulada")
print("="*60)

# Inicializa a base de dados
db = PatientDatabase()

# Lista pacientes
print("\n📋 Pacientes disponíveis:")
print("-"*60)
pacientes = db.list_patients_brief()
for p in pacientes:
    print(f"ID {p['id']}: {p['nome']}, {p['idade']} anos, Sexo: {p['sexo']}")

# Consulta prontuário completo do primeiro paciente
print("\n📄 Prontuário completo (Paciente ID 1):")
print("-"*60)
prontuario = db.get_patient_by_id(1)
print(f"Nome: {prontuario['nome']}")
print(f"Idade: {prontuario['idade']} anos")
print(f"Sexo: {prontuario['sexo']}")
print(f"Tipo Sanguíneo: {prontuario['tipo_sanguineo']}")
print(f"Peso: {prontuario['peso_kg']} kg | Altura: {prontuario['altura_cm']} cm")
print(f"Histórico: {', '.join(prontuario['historico_medico'])}")
print(f"Medicações: {len(prontuario['medicacoes'])} medicamentos")
print(f"Alergias: {', '.join(prontuario['alergias'])}")

# Gera resumo para uso no LLM
print("\n📝 Resumo para injeção no LLM:")
print("-"*60)
resumo = db.get_patient_summary(1)
print(resumo[:800] + "...")

print("\n✅ Etapa 2 concluída!")

2026-03-18 19:28:34 | INFO     | src.database.patient_records | PatientDatabase inicializado em: /kaggle/working/tech-challenge-fase-3/src/database/prontuarios.db


🗄️ ETAPA 2: Base de Prontuários Simulada

📋 Pacientes disponíveis:
------------------------------------------------------------
ID 1: Maria Silva Santos, 45 anos, Sexo: F
ID 2: João Pedro Oliveira, 62 anos, Sexo: M
ID 3: Ana Beatriz Costa, 28 anos, Sexo: F
ID 4: Carlos Eduardo Ferreira, 55 anos, Sexo: M
ID 5: Lucia Helena Rodrigues, 72 anos, Sexo: F

📄 Prontuário completo (Paciente ID 1):
------------------------------------------------------------
Nome: Maria Silva Santos
Idade: 45 anos
Sexo: F
Tipo Sanguíneo: O+
Peso: 68.5 kg | Altura: 162.0 cm
Histórico: Hipertensão arterial sistêmica (diagnosticada em 2018), Diabetes mellitus tipo 2 (diagnosticada em 2020), Dislipidemia
Medicações: 3 medicamentos
Alergias: Dipirona, Sulfonamidas

📝 Resumo para injeção no LLM:
------------------------------------------------------------
PRONTUÁRIO DO PACIENTE:
Nome: Maria Silva Santos | Idade: 45 anos | Sexo: F
Peso: 68.5kg | Altura: 162.0cm | Tipo sanguíneo: O+
Histórico: Hipertensão arterial sistê

## 🧠 ETAPA 3: Fine-tuning do LLM (BioMistral/TinyLlama + LoRA)
**Objetivo:** Ajustar o modelo com os dados médicos usando LoRA para especialização.

- Utiliza LoRA/QLoRA para eficiência computacional
- Modelo base: BioMistral-7B (ou TinyLlama para testes rápidos)
- Treinamento com dados dos scrapers
- Avaliação com perplexidade e métricas de QA

In [4]:
from src.fine_tuning.training import ModelTrainer
from src.fine_tuning.evaluation import ModelEvaluator
import logging

# Configurações para o Colab (GPU T4)
os.environ["BASE_MODEL_NAME"] = "BioMistral/BioMistral-7B"
os.environ["NUM_EPOCHS"] = "2"
os.environ["BATCH_SIZE"] = "1"
os.environ["MAX_SEQ_LENGTH"] = "2048"

print("="*60)
print("🧠 ETAPA 3: Fine-tuning com LoRA")
print("="*60)
print(f"Modelo base: {os.environ['BASE_MODEL_NAME']}")
print(f"Épocas: {os.environ['NUM_EPOCHS']}, Batch: {os.environ['BATCH_SIZE']}")
print(f"Max Seq Length: {os.environ['MAX_SEQ_LENGTH']}")
print("-"*60)

trainer = ModelTrainer()
model, tokenizer = trainer.train(dataset, force_retrain=False)

print("\n📈 Avaliando o Modelo...")
evaluator = ModelEvaluator(model, tokenizer)
metrics = evaluator.evaluate(dataset)

print(f"\n✅ Etapa 3 concluída!")
print(f"Métricas: {metrics}")

2026-03-18 19:28:53.874379: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773862134.040720     160 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773862134.098604     160 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773862134.493222     160 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773862134.493266     160 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773862134.493269     160 computation_placer.cc:177] computation placer alr

🧠 ETAPA 3: Fine-tuning com LoRA
Modelo base: BioMistral/BioMistral-7B
Épocas: 2, Batch: 1
Max Seq Length: 2048
------------------------------------------------------------

⚠️  MODELO EXISTENTE DETECTADO

Já existe um modelo treinado em:
  models/final_model

O que você deseja fazer?
  [1] Sobrescrever o modelo existente e treinar novamente
  [2] Usar o modelo existente e pular o treinamento



Digite sua escolha (1 ou 2):  2


2026-03-18 19:29:25 | INFO     | src.fine_tuning.training | Usuário escolheu usar o modelo existente
2026-03-18 19:29:25 | INFO     | src.fine_tuning.training | Pulando treinamento - usando modelo existente
2026-03-18 19:29:25 | INFO     | src.fine_tuning.training | Carregando modelo existente de: models/final_model


config.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

2026-03-18 19:31:04 | INFO     | src.fine_tuning.training | Modelo existente com LoRA carregado com sucesso!
The model 'PeftModelForCausalLM' is not supported for text-generation. Supported models are ['BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'ElectraForCausalLM', 'ErnieForCausalLM', 'FalconForCausalLM', 'FalconMambaForCausalLM', 'FuyuForCausalLM', 'GemmaForCausalLM', 'Gemma2ForCausalLM', 'GitForCausalLM', 'GlmForCausalLM', 'GPT2LMHeadModel', 'GPT2LMHeadModel', 'GPTBigCodeForCausalLM', 'GPTNeoForCausalLM', 'GPTNeoXForCausalLM', 'GPTNeoXJapaneseForCausalLM', 'GPTJForCausalLM', 'GraniteForCausalLM', 'GraniteMoeForCausalLM', 'JambaForCausalLM', 'JetMoeF


📈 Avaliando o Modelo...


2026-03-18 19:32:16 | INFO     | src.fine_tuning.evaluation | Perplexidade: 2.18
2026-03-18 19:32:16 | INFO     | src.fine_tuning.evaluation | Testando geração de respostas médicas...
2026-03-18 19:32:52 | INFO     | src.fine_tuning.evaluation | 
Pergunta: Como estruturar um laudo de RM de Coluna Lombar?
2026-03-18 19:32:52 | INFO     | src.fine_tuning.evaluation | Resposta:  Protocolo: Estruturação do Laudo de Imagem Média (RM) na coluna lombar. Diretrizes Clínicas: Indicações, contraindicações e resultados finais.

### Contexto:
- Instituição: Hospital XYZ
- Data: 2021-08-05
- Indicação: Dor lombar persistente após tratamento conservador

### Resultado:
- Espaço epidural normal em nível T12-L3;
- Disco intervertebral L4-L5 com sinais inflamatórios moderados;
- Sinal de discopatia modesta no nível L5-S1;
- Corpo vertebral normal em todas as regiões avaliadas;
- Canal neural sem compressão;
- Aparência geral normal.

### Conclusão:
- Alterações degenerativas leves na coluna lombar infe


✅ Etapa 3 concluída!
Métricas: {'perplexity': 2.1812629602917726}


## 🔍 ETAPA 4: Sistema RAG (Retrieval-Augmented Generation)
**Objetivo:** Implementar busca semântica em protocolos médicos para enriquecer respostas.

- Indexação de documentos com TF-IDF (padrão) ou Embeddings
- Busca semântica de documentos relevantes
- Integração com o assistente para contextualização
- Citação de fontes nas respostas

In [5]:
from src.langchain_integration.rag import MedicalRAG

print("="*60)
print("🔍 ETAPA 4: Sistema RAG")
print("="*60)

# Inicializa o sistema RAG
rag = MedicalRAG(data_dir="./data")

# Estatísticas do índice
print("\n📊 Estatísticas do RAG:")
print("-"*60)
stats = rag.get_stats()
print(f"Documentos indexados: {stats['total_documents']}")
print(f"Tipo de índice: {stats['index_type']}")
print(f"Fontes: {list(stats['sources'].keys())}")

# Teste de busca
print("\n🔎 Teste de busca semântica:")
print("-"*60)
query = "como tratar diabetes mellitus"
results = rag.search(query, top_k=3)

print(f"Query: '{query}'")
print(f"Resultados encontrados: {len(results)}")
print("\nTop 3 documentos mais relevantes:")
for i, doc in enumerate(results, 1):
    score = doc.get('relevance_score', 0)
    fonte = doc.get('source', 'Desconhecida')
    texto = doc.get('text', '')[:200]
    print(f"\n{i}. Score: {score:.4f}")
    print(f"   Fonte: {fonte}")
    print(f"   Conteúdo: {texto}...")

print("\n✅ Etapa 4 concluída!")

2026-03-18 19:34:08 | INFO     | src.langchain_integration.rag | Carregados 77 documentos de 3 arquivos
2026-03-18 19:34:08 | INFO     | src.langchain_integration.rag | Índice TF-IDF criado com sucesso
2026-03-18 19:34:08 | INFO     | src.langchain_integration.rag | MedicalRAG inicializado com 77 documentos. Método: TF-IDF


🔍 ETAPA 4: Sistema RAG

📊 Estatísticas do RAG:
------------------------------------------------------------
Documentos indexados: 77
Tipo de índice: tfidf
Fontes: ['TelessaúdeRS-UFRGS', 'CONITEC/MS', 'RadReport-RSNA']

🔎 Teste de busca semântica:
------------------------------------------------------------
Query: 'como tratar diabetes mellitus'
Resultados encontrados: 2

Top 3 documentos mais relevantes:

1. Score: 0.2795
   Fonte: CONITEC/MS
   Conteúdo: Quais são as diretrizes do protocolo de Diabetes Insípido? Especialidade: Endocrinologia Protocolo para diagnóstico diferencial e tratamento do diabetes insípido central e nefrogênico....

2. Score: 0.1072
   Fonte: TelessaúdeRS-UFRGS
   Conteúdo: Qual o manejo inicial do paciente com diabetes tipo 2 recém-diagnosticado? Especialidade: Endocrinologia / Atenção Primária | Categoria: Manejo clínico 1) Confirmar diagnóstico (glicemia jejum ≥126 ou...

✅ Etapa 4 concluída!


## 🔄 ETAPA 5: Workflow LangGraph
**Objetivo:** Orquestrar o fluxo de conversação com máquina de estados.

- Classificação automática de mensagens (saudação, emergência, pergunta, despedida)
- Roteamento condicional para handlers especializados
- Estado compartilhado entre nós do grafo
- Extensível para novos tipos de mensagem

In [6]:
from src.langgraph_flows.medical_workflow import MedicalWorkflow
from src.langchain_integration.assistant import MedicalAssistant

print("="*60)
print("🔄 ETAPA 5: Workflow LangGraph")
print("="*60)

# Inicializa o assistente (sem RAG por enquanto para teste do workflow)
assistant = MedicalAssistant(model=model, tokenizer=tokenizer, enable_rag=False)

# Cria o workflow
workflow = MedicalWorkflow(assistant)

# Testes de processamento de mensagens
print("\n📝 Testes de Processamento pelo Workflow:")
print("-"*60)

test_messages = [
    ("Olá, bom dia!", "Saudação"),
    ("Estou com dor no peito e falta de ar", "Emergência"),
    ("Quais os sintomas da diabetes?", "Pergunta médica"),
    ("Minha temperatura está 38.5°C", "Sinais vitais"),
    ("Obrigado, até logo!", "Despedida")
]

for msg, tipo_esperado in test_messages:
    print(f"\n👤 Entrada: '{msg}'")
    print(f"   Tipo esperado: {tipo_esperado}")
    response = workflow.process(msg)
    print(f"🏥 Resposta: {response[:200]}...")
    print("-"*60)

print("\n✅ Etapa 5 concluída!")

2026-03-18 19:34:24 | INFO     | src.utils.validators | InputValidator inicializado
2026-03-18 19:34:24 | INFO     | src.langchain_integration.chains | MedicalChains inicializado para assistente generalista
2026-03-18 19:34:24 | INFO     | src.langchain_integration.tools | MedicalTools inicializado para assistente generalista
2026-03-18 19:34:24 | INFO     | src.database.patient_records | PatientDatabase inicializado em: /kaggle/working/tech-challenge-fase-3/src/database/prontuarios.db
2026-03-18 19:34:24 | INFO     | src.langchain_integration.assistant | Base de prontuários inicializada
2026-03-18 19:34:24 | INFO     | src.langchain_integration.assistant | MedicalAssistant (BioMistral) inicializado
2026-03-18 19:34:24 | INFO     | src.langchain_integration.tools | MedicalTools inicializado para assistente generalista
2026-03-18 19:34:24 | INFO     | src.langgraph_flows.medical_workflow | MedicalWorkflow (Generalista) inicializado
2026-03-18 19:34:24 | WARNING  | src.langchain_integrat

🔄 ETAPA 5: Workflow LangGraph

📝 Testes de Processamento pelo Workflow:
------------------------------------------------------------

👤 Entrada: 'Olá, bom dia!'
   Tipo esperado: Saudação
🏥 Resposta: Olá! 👋 Sou seu assistente virtual médico generalista.

Posso ajudá-lo com:
• Informações sobre sintomas e condições de saúde
• Orientações gerais de prevenção e bem-estar
• Dicas de quando procurar at...
------------------------------------------------------------

👤 Entrada: 'Estou com dor no peito e falta de ar'
   Tipo esperado: Emergência
🏥 Resposta: 🚨 ALERTA DE EMERGÊNCIA

Identifiquei que sua mensagem pode indicar uma situação urgente.

AÇÕES IMEDIATAS:
1. 📞 Ligue 192 (SAMU) ou 193 (Bombeiros)
2. 🏥 Vá ao pronto-socorro mais próximo
3. ❌ NÃO diri...
------------------------------------------------------------

👤 Entrada: 'Quais os sintomas da diabetes?'
   Tipo esperado: Pergunta médica
🏥 Resposta: Sintomas: Polidipsia, polyuria, perda de peso, fadiga, cefaleias, neuropatia diabética,

## 🤖 ETAPA 6: Assistente Médico Completo
**Objetivo:** Integrar todas as funcionalidades em um assistente completo.

- LangChain para orquestração de chains
- RAG para busca em protocolos médicos
- Base de prontuários para contexto personalizado
- Citação de fontes (explainability)
- Detecção de emergências
- Validação de segurança de inputs

In [7]:
from src.langchain_integration.assistant import MedicalAssistant

print("="*60)
print("🤖 ETAPA 6: Assistente Médico Completo")
print("="*60)

# Inicializa assistente completo com RAG e paciente
print("\n🔧 Inicializando assistente com RAG e prontuário do paciente 1...")
assistant_completo = MedicalAssistant(
    model=model,
    tokenizer=tokenizer,
    enable_rag=True,
    patient_id=1  # Maria Silva Santos - Diabetes e Hipertensão
)

def chat_com_assistente(pergunta, mostrar_fontes=True):
    """Função auxiliar para interagir com o assistente"""
    print(f"\n👤 Usuário: {pergunta}")
    print("-"*60)
    resposta = assistant_completo.process_message(pergunta)
    print(f"🏥 Assistente: {resposta}")
    return resposta

# Testes de diferentes categorias
print("\n" + "="*60)
print("TESTES DO ASSISTENTE COMPLETO")
print("="*60)

# 1. Teste com RAG e contexto do paciente
print("\n📋 TESTE 1: Pergunta sobre condição do paciente (com RAG)")
chat_com_assistente("Quais cuidados devo ter com minha diabetes?")

# 2. Teste de estruturação de laudo
print("\n📋 TESTE 2: Estruturação de laudo radiológico")
chat_com_assistente("Como estruturar um laudo de TC de Crânio?")

# 3. Teste de emergência
print("\n📋 TESTE 3: Detecção de emergência")
chat_com_assistente("Estou com dor no peito e falta de ar!")

# 4. Teste de temperatura
print("\n📋 TESTE 4: Avaliação de sinais vitais")
chat_com_assistente("Minha temperatura está 38.8°C, o que fazer?")

# 5. Teste de conhecimento geral
print("\n📋 TESTE 5: Conhecimento médico geral")
chat_com_assistente("Qual a diferença entre gripe e resfriado?")

print("\n✅ Etapa 6 concluída!")

2026-03-18 19:35:27 | INFO     | src.utils.validators | InputValidator inicializado
2026-03-18 19:35:27 | INFO     | src.langchain_integration.chains | MedicalChains inicializado para assistente generalista
2026-03-18 19:35:27 | INFO     | src.langchain_integration.tools | MedicalTools inicializado para assistente generalista
2026-03-18 19:35:27 | INFO     | src.langchain_integration.rag | Carregados 77 documentos de 3 arquivos
2026-03-18 19:35:27 | INFO     | src.langchain_integration.rag | Índice TF-IDF criado com sucesso
2026-03-18 19:35:27 | INFO     | src.langchain_integration.rag | MedicalRAG inicializado com 77 documentos. Método: TF-IDF
2026-03-18 19:35:27 | INFO     | src.langchain_integration.assistant | RAG inicializado: {'total_documents': 77, 'index_type': 'tfidf', 'sources': {'TelessaúdeRS-UFRGS': 15, 'CONITEC/MS': 50, 'RadReport-RSNA': 12}, 'data_dir': 'data/raw'}
2026-03-18 19:35:27 | INFO     | src.database.patient_records | PatientDatabase inicializado em: /kaggle/wor

🤖 ETAPA 6: Assistente Médico Completo

🔧 Inicializando assistente com RAG e prontuário do paciente 1...

TESTES DO ASSISTENTE COMPLETO

📋 TESTE 1: Pergunta sobre condição do paciente (com RAG)

👤 Usuário: Quais cuidados devo ter com minha diabetes?
------------------------------------------------------------
🏥 Assistente: Cuidados para controle da pressão arterial, colesterol e glicemia. Evitar tabaco e álcool. Realizar exames periódicos. Utilizar medicamentos conforme orientação do médico. Adotar hábitos saudáveis.
📚 **Fontes consultadas:**
  [1] Prontuário do Paciente - Base de dados interna de prontuários
  [2] CONITEC/MS - Quais são as diretrizes do protocolo de Diabetes Insípido?
  [3] CONITEC/MS - Quais são as diretrizes do protocolo de Esclerose Lateral Amiotrófica?
  [4] TelessaúdeRS-UFRGS - Qual o manejo inicial do paciente com diabetes tipo 2 recém-diagnosticado?

📋 TESTE 2: Estruturação de laudo radiológico

👤 Usuário: Como estruturar um laudo de TC de Crânio?
--------------

2026-03-18 19:36:13 | WARNING  | src.langchain_integration.tools | Possível emergência detectada: 'dor no peito'


🏥 Assistente: ACHADOS: Parênquima cerebral: Hematoma subcapsular frontal direito acentuadamente hiperdenso na região posterior, hipotensilência da matriz cerebral, áreas de hipoperfusão difusa nos lobos temporais e parietais. Sistema ventricular: Sistema ventricular não dilatado, septo interventricular intacto. Cisternas: Espaço subaracnoide normal. Linha média: Desviada para esquerda. Estruturas ósseas: Integridade ótima. Seios paranosais e mastoides: Aéreados. IMPRESSION: Lesão hemorrágica supratentorial frontal direita, possivelmente isquêmica.
📚 **Fontes consultadas:**
  [1] Prontuário do Paciente - Base de dados interna de prontuários
  [2] RadReport-RSNA - Como estruturar um laudo de TC de Crânio sem Contraste?
  [3] RadReport-RSNA - Como estruturar um laudo de TC de Tórax com Contraste?
  [4] RadReport-RSNA - Como estruturar um laudo de TC de Abdome Total?

📋 TESTE 3: Detecção de emergência

👤 Usuário: Estou com dor no peito e falta de ar!
---------------------------------------

## 🏁 Conclusão da Pipeline Completa

O pipeline completa foi executado com sucesso, cobrindo todas as etapas do projeto:

### ✅ Resumo das Etapas

| Etapa | Descrição | Status |
|-------|-----------|--------|
| 1 | Coleta e preparação de dados (Scraping + Anonimização) | ✅ |
| 2 | Base de prontuários simulada (SQLite) | ✅ |
| 3 | Fine-tuning do LLM (BioMistral/TinyLlama + LoRA) | ✅ |
| 4 | Sistema RAG (Retrieval-Augmented Generation) | ✅ |
| 5 | Workflow LangGraph (orquestração de estados) | ✅ |
| 6 | Assistente médico completo (integração total) | ✅ |

### 📁 Arquivos Gerados

- `data/processed/medical_data_unified.jsonl`: Dataset unificado e anonimizado
- `models/final_model/`: Pesos do modelo após fine-tuning
- `logs/`: Logs detalhados do sistema

---
*Tech Challenge IADT - Fase 3 | FIAP/Alura*